# Bloco 00 Frontend Separado da logica

*   Script  html
*   Script css
*   Script js



In [ ]:
%%writefile interface.html
UI_HTML = <style>
  :root { --primary: #e63946; --secondary: #2a9d8f; --bg: #121212; --card: #1e1e1e; }
  .translator-card {
    background: var(--card); padding: 25px; border-radius: 20px; color: #f0f0f0;
    font-family: 'Segoe UI', Roboto, sans-serif; max-width: 450px; width: 90%;
    margin: 20px auto; box-shadow: 0 10px 30px rgba(0,0,0,0.5); border: 1px solid #333;
  }
  .header { margin-bottom: 20px; font-weight: 600; display: flex; align-items: center; justify-content: center; gap: 10px; }
  .controls { display: flex; gap: 10px; justify-content: center; margin-bottom: 20px; }

  button {
    padding: 12px 24px; border-radius: 12px; border: none; cursor: pointer;
    font-weight: bold; transition: all 0.3s ease; display: flex; align-items: center; gap: 8px;
  }
  #rec { background: var(--primary); color: white; }
  #rec:hover:not(:disabled) { transform: scale(1.05); filter: brightness(1.2); }
  #rec:disabled { background: #444; cursor: not-allowed; opacity: 0.6; }

  #stop { background: #333; color: #ccc; }
  #stop:not(:disabled) { background: #444; color: white; border: 1px solid var(--primary); }

  .output-area { background: #181818; padding: 18px; border-radius: 15px; border-left: 4px solid var(--secondary); }
  .label { font-size: 0.75rem; color: #777; text-transform: uppercase; letter-spacing: 1px; margin-bottom: 5px; display: block; }
  .text-content { margin: 0 0 15px 0; font-size: 1rem; line-height: 1.5; min-height: 24px; }

  .recording-pulse {
    width: 10px; height: 10px; background: var(--primary); border-radius: 50%;
    display: inline-block; animation: pulse 1.5s infinite;
  }
  @keyframes pulse { 0% { opacity: 1; } 50% { opacity: 0.3; } 100% { opacity: 1; } }
</style>

<div class="translator-card">
  <div class="header">🎙️ <span>Tradutor Inteligente</span></div>

  <div class="controls">
    <button id="rec" onclick="start()"><span>🔴</span> Gravar</button>
    <button id="stop" onclick="stop()" disabled><span>■</span> Parar</button>
  </div>

  <div class="output-area">
    <span class="label">Transcrição Original</span>
    <p id="p_txt" class="text-content">Aguardando voz...</p>

    <span class="label">Tradução em Tempo Real</span>
    <p id="p_tr" class="text-content" style="color: var(--secondary); font-weight: 500;">...</p>
  </div>

  <div id="status" style="font-size: 12px; color: #888; margin-top: 15px; text-align: center;">
    Sistema Pronto
  </div>
</div>

<script>
let recorder, chunks = [];
async function start() {
  const s = await navigator.mediaDevices.getUserMedia({ audio: true });
  recorder = new MediaRecorder(s);
  chunks = [];
  recorder.ondataavailable = e => chunks.push(e.data);
  recorder.onstop = async () => {
    const b = new Blob(chunks);
    const r = new FileReader();
    r.readAsDataURL(b);
    r.onloadend = () => google.colab.kernel.invokeFunction('callback_rec', [r.result], {});
  };
  recorder.start();
  document.getElementById('rec').disabled = true;
  document.getElementById('stop').disabled = false;
  document.getElementById('status').innerHTML = '<span class="recording-pulse"></span> Gravando Áudio...';
}

function stop() {
  recorder.stop();
  document.getElementById('stop').disabled = true;
  document.getElementById('rec').disabled = false;
  document.getElementById('status').innerText = "⏳ Processando tradução...";
}
</script>



Overwriting interface.html


- teste de api

In [ ]:
for m in client.models.list():
    print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-exp-image-generation
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-pro-preview-12-2025
models/ge

# Bloco 1: Setup e Dependências
Neste bloco instalamos as dependencias é essencial a execucao para o funcionamento e integralidade do Agente

In [ ]:
!pip install -U -q google-genai openai-whisper gTTS
import os, whisper, base64
from google.colab import userdata, output
from google import genai
from google.genai import types
from gtts import gTTS
from IPython.display import HTML, Audio, display
import logging
import tempfile
from requests.exceptions import ConnectionError
from typing import Tuple, Optional
from google.colab import output
from google.genai import errors


# Configuração do Cliente Gemini 2.0 (SDK 2026)
client = genai.Client(api_key=userdata.get('GOOGLE_API_KEY'))
model_whisper = whisper.load_model("base")

## Bloco 2: Backend (Lógica de Processamento)
Funções puras em Python para transcrição, tradução e síntese.

1.   Decodificacao do audio
2.   Transcricao com Whisper
2.   Traducao com Api do GEMINI
3.   Preparacao com Parsing e sintetizacao com gTTS



In [ ]:
def pipeline_processamento(audio_base64):
    try:
        # 1. Decodificação do Áudio
        dados_audio = base64.b64decode(audio_base64.split(",")[1])
        arquivo_entrada = "entrada.wav"
        with open(arquivo_entrada, "wb") as f:
            f.write(dados_audio)

        # 2. Transcrição (Whisper)
        resultado_transcricao = model_whisper.transcribe(arquivo_entrada, fp16=False, task="transcribe")
        texto_original = resultado_transcricao["text"].strip()

       # 3. Tradução com Lógica de Inversão (Gemini)
        instrucao_sistema = """Você é um tradutor automático rigoroso.
        Sua saída deve seguir EXATAMENTE o formato: codigo|texto

        Regras de detecção:
        - Se o usuário falar em Português -> Retorne: en|tradução para inglês
        - Se o usuário falar em Inglês -> Retorne: pt|tradução para português

        Exemplos de saída esperada:
        en|Good morning, how can I help you?
        pt|Bom dia, como posso ajudar?

        NUNCA adicione explicações, aspas ou textos extras. Apenas 'codigo|texto'."""
        try:
            resposta = client.models.generate_content(
                model='gemini-2.0-flash', # Versão atual estável
                contents=f"Traduza: {texto_original}",
                config=types.GenerateContentConfig(
                    system_instruction=instrucao_sistema,
                    temperature=0.1
                )
            )
            saida_bruta = resposta.text.strip()
        except Exception as erro_api:
            if "429" in str(erro_api):
                return texto_original, "Erro: Limite atingido. Aguarde 1 min.", None
            raise erro_api

        # 4. Parsing e Síntese de Voz (gTTS)
        if "|" in saida_bruta:
            codigo_idioma, texto_traduzido = [s.strip() for s in saida_bruta.split("|", 1)]
            idioma_voz = "pt" if "pt" in codigo_idioma.lower() else "en"
        else:
            texto_traduzido = saida_bruta
            idioma_voz = "pt"

        arquivo_saida = "saida.mp3"
        sintetizador = gTTS(text=texto_traduzido, lang=idioma_voz)
        sintetizador.save(arquivo_saida)

        return texto_original, texto_traduzido, arquivo_saida

    except Exception as erro:
        return f"Erro: {str(erro)}", "Falha no processamento", None

##  Bloco 3: Chamando o front e Integrando com Python
Este bloco chama a interface visual e integra a logica de processamento  Multimoldal



In [ ]:


def carregar_interface():
    with open('interface.html', 'r', encoding='utf-8') as f:
        return f.read()

UI_HTML = carregar_interface()

def processar_gravacao(resultado_audio):
    # t = texto_original, tr = texto_traduzido
    texto_original, texto_traduzido, caminho_audio = pipeline_processamento(resultado_audio)

    # Atualiza a interface usando os IDs do HTML
    output.eval_js(f"document.getElementById('p_txt').innerText = {repr(texto_original)}")
    output.eval_js(f"document.getElementById('p_tr').innerText = {repr(texto_traduzido)}")
    output.eval_js("document.getElementById('status').innerText = '✅ Concluído'")

    if caminho_audio:
        display(Audio(caminho_audio, autoplay=True))

# Registra a função com o novo nome
output.register_callback('callback_rec', processar_gravacao)
display(HTML(UI_HTML))